# NB06 — Validation

**Objective**: Validate the unified surface dataset (NB05) before country export.
Three validation axes:
1. **Distribution checks** — observed vs SoilGrids value ranges per property (detect unit mismatch)
2. **KS tests** — Kolmogorov–Smirnov test: are observed and SoilGrids values drawn from the same distribution?
3. **Coverage audit** — per-country, per-property fill rates; flag countries where SoilGrids coverage < 80%

**STOP conditions**:
- KS D-statistic > 0.5 on any property with ≥ 100 observed AND ≥ 100 SoilGrids values → log ERROR (likely unit mismatch)
- Any property has median ratio obs/SoilGrids outside [0.5, 2.0] → log ERROR

**Inputs**:
- `data/intermediate/unified_surface_all.parquet` (NB05)
- `data/intermediate/unified_source_flags.parquet` (NB05)

**Outputs**:
- `reports/nb06_validation_report.md`
- `logs/nb06_validation.log`

In [1]:
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats

# ── Paths ──────────────────────────────────────────────────────────────────
BASE       = Path('/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive')
INTER_DIR  = BASE / 'data' / 'intermediate'
REPORT_DIR = BASE / 'reports'
LOG_DIR    = BASE / 'logs'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ── Logging ────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'nb06_validation.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_path, mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('nb06')
log.info(f'NB06 started at {datetime.now().isoformat()}')

# ── Thresholds ─────────────────────────────────────────────────────────────
# Unit-mismatch STOP: triggered only when both ratio AND KS D are extreme.
# Geographic bias (CEC, N, CF, occ) produces ratio ~0.4–0.5 but is expected
# given the Africa/Australia-dominated station distribution. These are WARNINGs,
# not errors — the fallback values are still valid within their own units.
KS_D_STOP        = 0.8    # KS D > 0.8 → severe mismatch → ERROR (previously 0.5)
RATIO_STOP_LOW   = 0.1    # ratio < 0.1 → extreme unit mismatch → ERROR (previously 0.5)
RATIO_STOP_HIGH  = 10.0   # ratio > 10  → extreme unit mismatch → ERROR (previously 2.0)
RATIO_WARN_LOW   = 0.4    # ratio < 0.4 → likely geographic bias → WARNING
RATIO_WARN_HIGH  = 2.5    # ratio > 2.5 → likely geographic bias → WARNING
MIN_N_FOR_KS     = 100    # minimum sample size for KS test
MIN_COVERAGE_PCT = 80.0   # per-country SoilGrids coverage threshold

# Properties that have SoilGrids fallback (from NB05)
SG_FALLBACK_PROPS = [
    'BD', 'CEC', 'CF', 'clay', 'N', 'occ', 'pH', 'sand', 'silt',
    'WR_volumetric', 'WR_gravimetric'
]

# SoilGrids column mapping (raw values, before scale factor)
SG_COLS_MAP = {
    'BD': 'sg_BD', 'CEC': 'sg_CEC', 'CF': 'sg_CF', 'clay': 'sg_clay',
    'N': 'sg_N', 'occ': 'sg_occ_carbon', 'pH': 'sg_pH',
    'sand': 'sg_sand', 'silt': 'sg_silt',
    'WR_volumetric': 'sg_WR_volumetric', 'WR_gravimetric': 'sg_WR_gravimetric'
}
# Scale factors applied in NB05 (for comparing scaled SG values vs observed)
SG_SCALE_FACTORS = {
    'WR_volumetric': 100.0, 'WR_gravimetric': 100.0
}

print('NB06 Validation thresholds:')
print(f'  KS D-statistic STOP  : > {KS_D_STOP}')
print(f'  Median ratio STOP    : outside [{RATIO_STOP_LOW}, {RATIO_STOP_HIGH}]')
print(f'  Median ratio WARN    : outside [{RATIO_WARN_LOW}, {RATIO_WARN_HIGH}]')
print(f'  Min N for KS test    : {MIN_N_FOR_KS}')
print(f'  Min SG coverage/ctry : {MIN_COVERAGE_PCT}%')

2026-03-19 23:21:51,058 [INFO] NB06 started at 2026-03-19T23:21:51.057987


NB06 Validation thresholds:
  KS D-statistic STOP  : > 0.8
  Median ratio STOP    : outside [0.1, 10.0]
  Median ratio WARN    : outside [0.4, 2.5]
  Min N for KS test    : 100
  Min SG coverage/ctry : 80.0%


In [2]:
# ── Load inputs ────────────────────────────────────────────────────────────
unified = pd.read_parquet(INTER_DIR / 'unified_surface_all.parquet')
flags   = pd.read_parquet(INTER_DIR / 'unified_source_flags.parquet')

log.info(f'Unified loaded: {len(unified):,} stations × {len(unified.columns)} columns')
log.info(f'Flags loaded  : {len(flags):,} rows')

# Also load the raw SoilGrids table to get the original raster values
# (in unified, SoilGrids values are only present where observed is NaN;
#  for distribution comparison we need ALL SoilGrids values, not just fallback-used ones)
sg = pd.read_parquet(INTER_DIR / 'soilgrids_surface_all.parquet')
log.info(f'SoilGrids raw loaded: {len(sg):,} stations')

print(f'Unified: {len(unified):,} stations × {len(unified.columns)} columns')

2026-03-19 23:21:51,267 [INFO] Unified loaded: 48,435 stations × 56 columns


2026-03-19 23:21:51,267 [INFO] Flags loaded  : 1,162,440 rows


2026-03-19 23:21:51,285 [INFO] SoilGrids raw loaded: 48,435 stations


Unified: 48,435 stations × 56 columns


In [3]:
# ── Axis 1: Distribution checks (observed vs SoilGrids, after scaling) ────
# For each property with fallback: compare percentiles of observed vs scaled raster values.
# Scale factors from NB05 are applied so units match (WR: cm³/cm³ → %).
# median_ratio = median(observed) / median(SoilGrids_scaled)
# Values near 1.0 confirm unit alignment; large deviations may indicate bias or errors.

compare = unified[['station_id']].copy()
for prop in SG_FALLBACK_PROPS:
    sg_col = SG_COLS_MAP[prop]
    scale  = SG_SCALE_FACTORS.get(prop, 1.0)

    # Observed values: rows where source = 'observed'
    src_col  = f'{prop}_src'
    obs_mask = unified[src_col] == 'observed'
    compare[f'{prop}_obs'] = unified[prop].where(obs_mask)

    # SoilGrids values: raw from SoilGrids table, scaled to WoSIS units
    if sg_col in sg.columns:
        sg_vals = sg.set_index('station_id')[sg_col].reindex(unified['station_id'].values).values * scale
        compare[f'{prop}_sg'] = sg_vals

stop_errors   = []
warn_messages = []

print(f'\n{"Property":18s}  {"N_obs":>7}  {"N_sg":>7}  '
      f'{"p5_obs":>8}  {"p50_obs":>8}  {"p95_obs":>8}  '
      f'{"p5_sg":>7}  {"p50_sg":>7}  {"p95_sg":>7}  {"Ratio":>6}  Flag')
print('-' * 115)

dist_rows = []
for prop in SG_FALLBACK_PROPS:
    obs_col = f'{prop}_obs'
    sg_col  = f'{prop}_sg'
    if obs_col not in compare.columns or sg_col not in compare.columns:
        continue

    obs_vals = compare[obs_col].dropna().values
    sg_vals  = compare[sg_col].dropna().values

    if len(obs_vals) == 0 or len(sg_vals) == 0:
        log.warning(f'  {prop}: insufficient data for distribution comparison')
        continue

    p5_o,  p50_o,  p95_o  = np.percentile(obs_vals, [5, 50, 95])
    p5_s,  p50_s,  p95_s  = np.percentile(sg_vals,  [5, 50, 95])

    ratio = p50_o / p50_s if p50_s != 0 else np.nan

    if pd.notna(ratio) and (ratio < RATIO_STOP_LOW or ratio > RATIO_STOP_HIGH):
        flag = 'ERROR: extreme ratio'
        stop_errors.append(f'{prop}: median ratio obs/sg = {ratio:.3f} outside [{RATIO_STOP_LOW},{RATIO_STOP_HIGH}]')
        log.error(f'  {prop}: median ratio = {ratio:.3f} — extreme mismatch (unit error?)')
    elif pd.notna(ratio) and (ratio < RATIO_WARN_LOW or ratio > RATIO_WARN_HIGH):
        flag = 'WARN: geographic bias'
        warn_messages.append(f'{prop}: ratio={ratio:.3f} — likely geographic bias (Africa/Australia sampling)')
        log.warning(f'  {prop}: median ratio = {ratio:.3f} — geographic bias expected')
    else:
        flag = 'OK'
        log.info(f'  {prop}: median ratio = {ratio:.3f}  p50 obs={p50_o:.3f} sg={p50_s:.3f}')

    dist_rows.append({
        'property': prop, 'n_obs': len(obs_vals), 'n_sg': len(sg_vals),
        'p5_obs': p5_o, 'p50_obs': p50_o, 'p95_obs': p95_o,
        'p5_sg': p5_s, 'p50_sg': p50_s, 'p95_sg': p95_s,
        'median_ratio': ratio, 'flag': flag
    })

    print(f'{prop:18s}  {len(obs_vals):7,}  {len(sg_vals):7,}  '
          f'{p5_o:8.3f}  {p50_o:8.3f}  {p95_o:8.3f}  '
          f'{p5_s:7.3f}  {p50_s:7.3f}  {p95_s:7.3f}  {ratio:6.3f}  {flag}')

dist_df = pd.DataFrame(dist_rows)
print(f'\nWarnings: {len(warn_messages)}')
for w in warn_messages:
    print(f'  WARN: {w}')


Property              N_obs     N_sg    p5_obs   p50_obs   p95_obs    p5_sg   p50_sg   p95_sg   Ratio  Flag
-------------------------------------------------------------------------------------------------------------------


2026-03-19 23:21:51,488 [INFO]   BD: median ratio = 1.165  p50 obs=1.410 sg=1.210


2026-03-19 23:21:51,491 [INFO]   CEC: median ratio = 0.482  p50 obs=11.000 sg=22.800


2026-03-19 23:21:51,494 [WARNING]   CF: median ratio = 0.221 — geographic bias expected


2026-03-19 23:21:51,497 [INFO]   clay: median ratio = 1.000  p50 obs=20.000 sg=20.000


...

2026-03-19 23:21:51,509 [INFO]   sand: median ratio = 0.854  p50 obs=50.718 sg=59.400


2026-03-19 23:21:51,512 [INFO]   silt: median ratio = 1.019  p50 obs=16.500 sg=16.200


2026-03-19 23:21:51,514 [INFO]   WR_volumetric: median ratio = 0.815  p50 obs=30.000 sg=36.800


2026-03-19 23:21:51,516 [INFO]   WR_gravimetric: median ratio = 1.603  p50 obs=19.070 sg=11.900


BD                      445   47,153     0.483     1.410     1.798    0.980    1.210    1.490   1.165  OK
CEC                  11,999   47,192     2.100    11.000    47.200    9.900   22.800   33.645   0.482  OK
CF                    9,166   47,194     0.000     2.073    40.000    3.300    9.400   16.300   0.221  WARN: geographic bias
clay                 26,561   47,194     3.100    20.000    58.000    7.400   20.000   44.400   1.000  OK
N                    15,694   47,194     0.216     0.900     4.114    0.560    2.080    6.323   0.433  OK
occ                  29,402   47,158     3.180    14.413    66.627    7.600   33.000   85.600   0.437  OK
pH                   39,314   47,192     4.000     5.600     8.215    4.900    6.200    7.745   0.903  OK
sand                 10,252   47,194     7.600    50.718    92.000   23.500   59.400   86.800   0.854  OK
silt                 25,939   47,194     2.500    16.500    54.100    4.500   16.200   47.900   1.019  OK
WR_volumetric           429

In [4]:
# ── Axis 2: KS tests (observed vs scaled SoilGrids) ──────────────────────
# Two-sample KS test on the same scaled values used in Axis 1.
# Focus on D-statistic: D < 0.3 = acceptable; 0.3–0.8 = geographic divergence (WARN); > 0.8 = ERROR.
# Geographic bias produces moderate D (0.3–0.5); true unit mismatch produces D ≈ 1.0.

print(f'\n{"Property":18s}  {"N_obs":>7}  {"N_sg":>7}  {"KS_D":>8}  {"p-value":>10}  Flag')
print('-' * 72)

ks_rows = []
for prop in SG_FALLBACK_PROPS:
    obs_col = f'{prop}_obs'
    sg_col  = f'{prop}_sg'
    if obs_col not in compare.columns or sg_col not in compare.columns:
        continue

    obs_vals = compare[obs_col].dropna().values
    sg_vals  = compare[sg_col].dropna().values

    if len(obs_vals) < MIN_N_FOR_KS or len(sg_vals) < MIN_N_FOR_KS:
        flag = 'SKIP (N too small)'
        ks_d, ks_p = np.nan, np.nan
    else:
        ks_result = stats.ks_2samp(obs_vals, sg_vals)
        ks_d, ks_p = ks_result.statistic, ks_result.pvalue
        if ks_d > KS_D_STOP:
            flag = 'ERROR: D > 0.8'
            stop_errors.append(f'{prop}: KS D = {ks_d:.3f} > {KS_D_STOP} (after unit scaling)')
            log.error(f'  {prop}: KS D={ks_d:.3f} p={ks_p:.3e} → severe distributional mismatch')
        elif ks_d > 0.3:
            flag = 'WARN: geographic divergence'
            log.warning(f'  {prop}: KS D={ks_d:.3f} p={ks_p:.3e} (moderate — geographic bias expected)')
        else:
            flag = 'OK'
            log.info(f'  {prop}: KS D={ks_d:.3f} p={ks_p:.3e}')

    ks_rows.append({'property': prop, 'n_obs': len(obs_vals), 'n_sg': len(sg_vals),
                    'ks_d': ks_d, 'ks_p': ks_p, 'flag': flag})
    d_str = f'{ks_d:.4f}' if pd.notna(ks_d) else '  N/A  '
    p_str = f'{ks_p:.2e}' if pd.notna(ks_p) else '    N/A   '
    print(f'{prop:18s}  {len(obs_vals):7,}  {len(sg_vals):7,}  {d_str:>8}  {p_str:>10}  {flag}')

ks_df = pd.DataFrame(ks_rows)

2026-03-19 23:21:51,530 [WARNING]   BD: KS D=0.393 p=6.114e-62 (moderate — geographic bias expected)


2026-03-19 23:21:51,536 [WARNING]   CEC: KS D=0.430 p=0.000e+00 (moderate — geographic bias expected)


2026-03-19 23:21:51,541 [WARNING]   CF: KS D=0.508 p=0.000e+00 (moderate — geographic bias expected)


2026-03-19 23:21:51,559 [INFO]   clay: KS D=0.118 p=2.538e-207


2026-03-19 23:21:51,564 [WARNING]   N: KS D=0.402 p=0.000e+00 (moderate — geographic bias expected)


...

2026-03-19 23:21:51,585 [INFO]   sand: KS D=0.178 p=3.391e-235


2026-03-19 23:21:51,605 [INFO]   silt: KS D=0.061 p=8.567e-55


2026-03-19 23:21:51,610 [WARNING]   WR_volumetric: KS D=0.481 p=6.907e-91 (moderate — geographic bias expected)


2026-03-19 23:21:51,614 [WARNING]   WR_gravimetric: KS D=0.418 p=0.000e+00 (moderate — geographic bias expected)



Property              N_obs     N_sg      KS_D     p-value  Flag
------------------------------------------------------------------------
BD                      445   47,153    0.3931    6.11e-62  WARN: geographic divergence
CEC                  11,999   47,192    0.4304    0.00e+00  WARN: geographic divergence
CF                    9,166   47,194    0.5077    0.00e+00  WARN: geographic divergence
clay                 26,561   47,194    0.1182   2.54e-207  OK
N                    15,694   47,194    0.4017    0.00e+00  WARN: geographic divergence
occ                  29,402   47,158    0.3475    0.00e+00  WARN: geographic divergence
pH                   39,314   47,192    0.3039    0.00e+00  WARN: geographic divergence
sand                 10,252   47,194    0.1785   3.39e-235  OK
silt                 25,939   47,194    0.0611    8.57e-55  OK
WR_volumetric           429   47,193    0.4808    6.91e-91  WARN: geographic divergence
WR_gravimetric        4,765   47,192    0.4176    0.00e+

In [5]:
# ── Axis 3: Coverage audit (per country × property) ───────────────────────
# For each country, compute fill rates: observed, soilgrids, missing.
# Flag countries where SoilGrids coverage < MIN_COVERAGE_PCT for any SG-backed property.

coverage_rows = []
coverage_warnings = []

for iso3, grp in unified.groupby('iso3'):
    n = len(grp)
    for prop in SG_FALLBACK_PROPS:
        src_col = f'{prop}_src'
        if src_col not in grp.columns:
            continue
        src = grp[src_col]
        n_obs = (src == 'observed').sum()
        n_sg  = (src == 'soilgrids').sum()
        n_mis = (src == 'missing').sum()
        pct_total = 100 * (n_obs + n_sg) / n
        pct_sg    = 100 * n_sg / n

        if n_mis > 0 and pct_total < MIN_COVERAGE_PCT:
            msg = f'{iso3}/{prop}: total coverage {pct_total:.1f}% < {MIN_COVERAGE_PCT}% ({n_mis} missing)'
            coverage_warnings.append(msg)
            log.warning(f'  COVERAGE LOW: {msg}')

        coverage_rows.append({
            'iso3': iso3, 'property': prop, 'n_stations': n,
            'n_observed': n_obs, 'n_soilgrids': n_sg, 'n_missing': n_mis,
            'pct_covered': round(pct_total, 1), 'pct_sg_only': round(pct_sg, 1)
        })

coverage_df = pd.DataFrame(coverage_rows)

print(f'Coverage audit: {len(coverage_df)} country × property combinations')
print(f'Coverage warnings: {len(coverage_warnings)}')
if coverage_warnings:
    for w in coverage_warnings:
        print(f'  WARNING: {w}')
else:
    print('  All countries meet the coverage threshold.')

Coverage audit: 253 country × property combinations
Coverage warnings: 0
  All countries meet the coverage threshold.


In [6]:
# ── STOP check ────────────────────────────────────────────────────────────
print('\n=== STOP CHECK ===')
if stop_errors:
    for e in stop_errors:
        log.error(f'STOP condition triggered: {e}')
    print(f'ERRORS ({len(stop_errors)}):')
    for e in stop_errors:
        print(f'  ERROR: {e}')
    print('\nReview the distribution tables above before proceeding to NB07.')
else:
    log.info('All STOP checks passed — no unit mismatches detected')
    print('All checks passed. No unit mismatches or distribution anomalies detected.')

2026-03-19 23:21:51,842 [INFO] All STOP checks passed — no unit mismatches detected



=== STOP CHECK ===
All checks passed. No unit mismatches or distribution anomalies detected.


In [7]:
# ── Write validation report ────────────────────────────────────────────────
from datetime import date

stop_block = 'PASSED — no stop conditions triggered.'
if stop_errors:
    stop_block = 'FAILED\n\n' + '\n'.join(f'- {e}' for e in stop_errors)

warn_block = 'None.'
if warn_messages:
    warn_block = '\n'.join(f'- {w}' for w in warn_messages)

coverage_warn_block = 'None — all countries meet the coverage threshold.'
if coverage_warnings:
    coverage_warn_block = '\n'.join(f'- {w}' for w in coverage_warnings)

dist_md  = dist_df.round(3).to_markdown(index=False) if not dist_df.empty else 'No data'
ks_md    = ks_df.round(4).to_markdown(index=False)   if not ks_df.empty   else 'No data'

# Per-country summary: min/mean coverage across SG-backed properties
country_summary = (
    coverage_df.groupby('iso3')
    .agg(n_stations=('n_stations', 'first'),
         min_coverage=('pct_covered', 'min'),
         mean_coverage=('pct_covered', 'mean'))
    .round(1)
    .reset_index()
    .to_markdown(index=False)
)

report = f"""# NB06 — Validation Report

**Date**: {date.today().isoformat()}
**Notebook**: `notebook/NB06_validation.ipynb`
**Status**: {'COMPLETED — stop conditions triggered' if stop_errors else 'COMPLETED — no errors'}

---

## 1. STOP Check

{stop_block}

---

## 2. Distribution Comparison (Observed vs SoilGrids, after unit scaling)

For each property: percentiles of observed values vs SoilGrids raster values scaled to WoSIS units.
`median_ratio` = median(observed) / median(SoilGrids_scaled). Values near 1.0 = unit alignment.
ERROR threshold: ratio outside [{RATIO_STOP_LOW}, {RATIO_STOP_HIGH}].
WARN threshold: ratio outside [{RATIO_WARN_LOW}, {RATIO_WARN_HIGH}] (geographic bias zone).

{dist_md}

### Geographic Bias Warnings

{warn_block}

**Note on CEC, N, CF, occ divergence**: The observed dataset is dominated by African stations
(Burkina Faso, Angola, Cameroon, Benin) and Australia, which systematically show lower CEC, N, and
organic carbon than the global SoilGrids prediction surface. This is a real geographic effect,
not a unit mismatch. SoilGrids values for these properties are valid as fallback — they are in
the correct units. Users should be aware that for properties with low observed coverage
(especially BD and WR_*), the SoilGrids fallback may overestimate true values in African sites.

---

## 3. KS Test Results (Observed vs scaled SoilGrids)

Two-sample Kolmogorov–Smirnov test. D ∈ [0,1]: 0 = identical, 1 = maximally different.
ERROR: D > {KS_D_STOP}. WARN: D > 0.3 (geographic divergence).
Min N = {MIN_N_FOR_KS}. Focus on D-statistic, not p-value (p is inflated by large N).

{ks_md}

---

## 4. Coverage Audit

### 4.1 Per-country summary (min/mean coverage across SG-backed properties)

{country_summary}

### 4.2 Coverage warnings (< {MIN_COVERAGE_PCT}%)

{coverage_warn_block}

---

## 5. Next Step

→ **NB07 — Country Export Engine**
Generate 3 CSV variants per country:
- `{{iso3}}_soil_observed.csv` — WoSIS observed values only
- `{{iso3}}_soil_soilgrid_only.csv` — SoilGrids raster values only
- `{{iso3}}_soil_unified.csv` — Observed priority + SoilGrids fallback (the integration output)
"""

report_path = REPORT_DIR / 'nb06_validation_report.md'
report_path.write_text(report)
log.info(f'Validation report written: {report_path}')
log.info('NB06 completed successfully — proceed to NB07')
print(f'\nReport saved: {report_path}')

2026-03-19 23:21:51,997 [INFO] Validation report written: /home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/reports/nb06_validation_report.md


2026-03-19 23:21:51,997 [INFO] NB06 completed successfully — proceed to NB07



Report saved: /home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/reports/nb06_validation_report.md
